# Portfolio Bot v3 — Hybrid: Monthly DCA + 1.5x ATH Sell + Progressive Dip Buying

**Combines best of both:**
- **Every month:** $20 BTC + $10 BNB (steady DCA, no signal filter)
- **On red candles:** Extra dip-buy from reserve using progressive reinvestment
- **At 1.5x ATH:** Sell 35% to lock in profits
- BTC progressive: 1% start, +2%/buy, cap 50% | BNB: 1% start, +1%/buy, cap 40%
- Dip buys happen **before** base DCA — priority to deploying reserve

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def fetch(s):
    url='https://api.binance.com/api/v3/klines'; ak=[]; st=0
    while True:
        p={'symbol': s, 'interval': '1M', 'startTime': st, 'limit': 500}
        d=requests.get(url, params=p).json()
        if not d: break
        ak.extend(d)
        if len(d)<500: break
        st=d[-1][0]+1; time.sleep(0.1)
    return ak
cols=['ts','o','h','l','c','v','ct','qv','t','tbb','tbq','ig']
btc=pd.DataFrame(fetch('BTCUSDT'), columns=cols); btc['date']=pd.to_datetime(btc['ts'], unit='ms', utc=True)
bnb=pd.DataFrame(fetch('BNBUSDT'), columns=cols); bnb['date']=pd.to_datetime(bnb['ts'], unit='ms', utc=True)
for c in ['o','h','l','c','v']: btc[c]=btc[c].astype(float); bnb[c]=bnb[c].astype(float)
btc=btc[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
bnb=bnb[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
s=max(btc['date'].min(), bnb['date'].min()); e=min(btc['date'].max(), bnb['date'].max())
btc=btc[(btc['date']>=s)&(btc['date']<=e)].reset_index(drop=True)
bnb=bnb[(bnb['date']>=s)&(bnb['date']<=e)].reset_index(drop=True)
print(f'BTC: {btc["date"].min().strftime("%Y-%m")} -> {btc["date"].max().strftime("%Y-%m")} ({len(btc)} mo)')
print(f'BNB: {bnb["date"].min().strftime("%Y-%m")} -> {bnb["date"].max().strftime("%Y-%m")} ({len(bnb)} mo)')

In [ ]:
SELL_R=0.35; SELL_M=1.5
usdt=0.0; inj=0.0; dip_total=0.0; base_total=0.0
bh=0.0; bhi=0.0; bhi_s=0.0; br=1.0
nh=0.0; nhi=0.0; nhi_s=0.0; nr=1.0
rec=[]

for i in range(len(btc)):
    b=btc.iloc[i]; n=bnb.iloc[i]
    bc=b['c']; nc=n['c']; d=b['date']; act=[]
    usdt+=30.0; inj+=30.0

    # Step 1: Progressive dip-buy on red candles (from reserve, before base buys)
    if bc<b['o'] and usdt>10:
        ext=min(usdt*(br/100.0), usdt-30)
        if ext>5:
            bh+=ext/bc; usdt-=ext; dip_total+=ext
            act.append(f'BTC dip ${ext:.0f} ({br:.0f}%)')
    if nc<n['o'] and usdt>10:
        ext=min(usdt*(nr/100.0), usdt-30)
        if ext>5:
            nh+=ext/nc; usdt-=ext; dip_total+=ext
            act.append(f'BNB dip ${ext:.0f} ({nr:.0f}%)')

    # Step 2: Monthly DCA base buys (every month, remaining cash)
    if usdt>=20:
        bh+=20.0/bc; usdt-=20.0; base_total+=20.0; act.append(f'BTC DCA $20 @ {bc:.0f}')
    if usdt>=10:
        nh+=10.0/nc; usdt-=10.0; base_total+=10.0; act.append(f'BNB DCA $10 @ {nc:.0f}')

    # Step 3: Update ATHs + sell at 1.5x
    if bc>bhi: bhi=bc
    if nc>nhi: nhi=nc

    if bh>0.000001 and bc>=bhi_s*SELL_M and bhi_s>0:
        s=bh*SELL_R; usdt+=s*bc; bh-=s; br=1.0; bhi_s=bhi
        act.append(f'BTC SELL 35% @ {bc:.0f}')
    elif bhi_s==0: bhi_s=bc
    else:
        if br<50: br=min(50, br+2)

    if nh>0.000001 and nc>=nhi_s*SELL_M and nhi_s>0:
        s=nh*SELL_R; usdt+=s*nc; nh-=s; nr=1.0; nhi_s=nhi
        act.append(f'BNB SELL 35% @ {nc:.0f}')
    elif nhi_s==0: nhi_s=nc
    else:
        if nr<40: nr=min(40, nr+1)

    rec.append({'date': d, 'bc': bc, 'nc': nc,
        'bh': bh, 'bv': bh*bc, 'br': br,
        'nh': nh, 'nv': nh*nc, 'nr': nr,
        'usdt': usdt, 'inj': inj, 'pf': bh*bc+nh*nc+usdt,
        'act': ' | '.join(act) if act else 'nothing'})

res=pd.DataFrame(rec)

In [ ]:
f=res.iloc[-1]
eq=res['pf']
ddr=(eq.cummax()-eq)/eq.cummax(); maxdd=ddr.max()*100
m=eq.pct_change().dropna(); m=m[m.index>=12]
sh=(m.mean()/m.std())*np.sqrt(12) if m.std()>0 else 0
p=m[m>0].sum(); ne=m[m<0].abs().sum(); pf_r=p/ne if ne>0 else float('inf')
ann=(1+(f['pf']/f['inj']-1))**(12/len(res))-1
cal=ann/(maxdd/100) if maxdd>0 else 0

sells=res[res['act'].str.contains('SELL', na=False)]
dip_mo=res[res['act'].str.contains('dip', na=False)]
btc_dips=res[res['act'].str.contains('BTC dip', na=False)]
bnb_dips=res[res['act'].str.contains('BNB dip', na=False)]

print('='*72)
print('PORTFOLIO BOT v3 — HYBRID')
print('Monthly DCA $20BTC+$10BNB + 1.5x ATH Sell 35% + Progressive Dip')
print('='*72)
print(f'Period:       {res["date"].min().strftime("%Y-%m")} -> {res["date"].max().strftime("%Y-%m")} ({len(res)} mo)')
print(f'Injection:    $30/mo = ${f["inj"]:.0f} total')
print(f'Dips funded by sell profits: ${dip_total:.0f}')
print()
print('--- ASSET BREAKDOWN ---')
btc_inj=base_total*20/30
bnb_inj=base_total*10/30
print(f'BTC:          ${f["bv"]:.2f} ({f["bh"]:.6f} @ {f["bc"]:.0f})')
print(f'BTC return:   {(f["bv"]/btc_inj-1)*100:.1f}%' if btc_inj>0 else 'BTC return:   N/A')
print(f'BNB:          ${f["nv"]:.2f} ({f["nh"]:.6f} @ {f["nc"]:.0f})')
print(f'BNB return:   {(f["nv"]/bnb_inj-1)*100:.1f}%' if bnb_inj>0 else 'BNB return:   N/A')
print(f'USDT reserve: ${f["usdt"]:.2f} ({(f["usdt"]/f["pf"])*100:.1f}% of portfolio)')
print()
print('--- PORTFOLIO ---')
print(f'Portfolio:    ${f["pf"]:.2f}')
print(f'Total P&L:    ${f["pf"]-f["inj"]:.2f}')
print(f'Return:       {(f["pf"]/f["inj"]-1)*100:.2f}%')
print(f'Max DD:       {maxdd:.2f}%')
print(f'Sharpe:       {sh:.2f}')
print(f'PF:           {pf_r:.2f}')
print(f'Calmar:       {cal:.2f}')
print()
print('--- CASHFLOW ---')
print(f'Base DCA:     ${base_total:.0f} (${base_total*20/30:.0f} BTC + ${base_total*10/30:.0f} BNB)')
print(f'Dip re-invest:${dip_total:.0f}')
print(f'Total spends: ${base_total+dip_total:.0f}')
print(f'Final reserve:${f["usdt"]:.0f}')
print(f'Idle ratio:   {(f["usdt"]/f["pf"])*100:.1f}%')
print()
print(f'SELLS: {len(sells)} events')
for _, r in sells.iterrows():
    a=r['act'].split(' | ')
    sp=[x for x in a if 'SELL' in x]
    print(f"  {r['date'].strftime('%Y-%m')}: {' | '.join(sp)}")
print()
print(f'DIP BUYS: {len(dip_mo)} months')
print(f'  BTC dip months: {len(btc_dips)}')
print(f'  BNB dip months: {len(bnb_dips)}')
print(f'  Total dipped:   ${dip_total:.0f}')

In [ ]:
fig=plt.figure(figsize=(16, 14))
gs=fig.add_gridspec(5, 2, height_ratios=[3, 1, 1.2, 1.2, 1.2])

ax=fig.add_subplot(gs[0, :])
ax.fill_between(res['date'], res['inj'], res['pf'],
    where=res['pf']>=res['inj'], color='green', alpha=0.12, label='Profit')
ax.fill_between(res['date'], res['pf'], res['inj'],
    where=res['pf']<res['inj'], color='red', alpha=0.12, label='Loss')
ax.plot(res['date'], res['inj'], color='gray', lw=1.5, ls='--', label='Injected ($30/mo)')
ax.plot(res['date'], res['pf'], color='#2563eb', lw=2, label='Portfolio')
ax.scatter(sells['date'], sells['pf'], color='#dc2626', s=50, marker='v', zorder=5, label='Sell 35%')
ax.set_title('Portfolio Bot v3 — Hybrid: Monthly DCA + 1.5x ATH Sell + Progressive Dip Buying', fontsize=12, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=8, ncol=3); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax=fig.add_subplot(gs[1, :])
ax.fill_between(res['date'], 0, ddr*100, color='#dc2626', alpha=0.35)
ax.plot(res['date'], ddr*100, color='#dc2626', lw=0.8)
ax.axhline(y=maxdd, color='#991b1b', ls=':', lw=1, label=f'Max DD: {maxdd:.1f}%')
ax.set_ylabel('DD (%)'); ax.set_ylim(bottom=0); ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax=fig.add_subplot(gs[2, :])
ax.stackplot(res['date'], res['bv'], res['nv'], res['usdt'],
    labels=['BTC','BNB','USDT'], colors=['#f59e0b','#8b5cf6','#10b981'], alpha=0.8)
ax.set_ylabel('Allocation'); ax.legend(fontsize=8, loc='upper left'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')); ax.set_ylim(bottom=0)

ax=fig.add_subplot(gs[3, 0])
ax.fill_between(res['date'], 0, res['bv'], color='#f59e0b', alpha=0.3)
ax.plot(res['date'], res['bv'], color='#d97706', lw=1.5)
ax_t=ax.twinx(); ax_t.step(res['date'], res['br'], where='post', color='#f59e0b', lw=0.8, alpha=0.5)
ax_t.set_ylabel('Reinvest %', color='#f59e0b'); ax_t.set_ylim(0, 58)
ax.set_ylabel('BTC Value'); ax.set_title('BTC'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax=fig.add_subplot(gs[3, 1])
ax.fill_between(res['date'], 0, res['nv'], color='#8b5cf6', alpha=0.3)
ax.plot(res['date'], res['nv'], color='#7c3aed', lw=1.5)
ax_t=ax.twinx(); ax_t.step(res['date'], res['nr'], where='post', color='#8b5cf6', lw=0.8, alpha=0.5)
ax_t.set_ylabel('Reinvest %', color='#8b5cf6'); ax_t.set_ylim(0, 46)
ax.set_ylabel('BNB Value'); ax.set_title('BNB'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax=fig.add_subplot(gs[4, 0])
ax.fill_between(res['date'], 0, res['usdt'], color='#10b981', alpha=0.3)
ax.plot(res['date'], res['usdt'], color='#059669', lw=1.5)
ax.set_ylabel('USDT'); ax.set_title('Reserve (deployed on red candles)'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax=fig.add_subplot(gs[4, 1])
ax.plot(res['date'], res['bc'], color='#f59e0b', lw=1, alpha=0.7, label='BTC')
axb=ax.twinx(); axb.plot(res['date'], res['nc'], color='#8b5cf6', lw=1, alpha=0.7, label='BNB')
ax.set_ylabel('BTC'); axb.set_ylabel('BNB'); ax.set_title('Prices')
ax.grid(True, alpha=0.25); ax.legend(loc='upper left', fontsize=8); axb.legend(loc='upper right', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-v3.png', dpi=150, bbox_inches='tight')
print('Saved dca-bt-buy-bot-portfolio-v3.png')
plt.show()